# LeetCode 695 — Max Area of Island bằng Python

Link đề: https://leetcode.com/problems/max-area-of-island/ · Độ khó: **Medium** · Chủ đề: Depth-First Search, Breadth-First Search, Union Find, Array, Matrix

Skeleton LeetCode:

```python
class Solution:
    def maxAreaOfIsland(self, grid: List[List[int]]) -> int:
        ...
```

Ghi chú: bài có **đáp án duy nhất** (một số nguyên) nên so sánh bằng `==`. Notebook so sánh 4 cách giải khác nhau về ý tưởng trên rất nhiều lưới ngẫu nhiên có seed, và kiểm tra cả các cài đặt sai phải bị bắt.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: lưới nhị phân `0/1`, một **đảo** là nhóm ô `1` nối nhau theo **4 hướng** (trên, dưới, trái, phải — **không** tính chéo); **diện tích** là số ô `1` của đảo; cần diện tích **lớn nhất**.
2. Nắm các công cụ Python: ma trận list-of-list, danh sách 4 hướng `DIRS`, kiểm tra biên `0 <= x < m`, `collections.deque`, đệ quy và `sys.setrecursionlimit`, `Counter`, Union-Find (DSU) có nén đường đi.
3. Đi từ **lan nhãn nhỏ nhất tới khi ổn định (brute force)** → **DFS đệ quy** → **BFS bằng hàng đợi (và DFS bằng ngăn xếp)** → **Union-Find đếm kích thước thành phần**.
4. Hiểu mấu chốt: đây là bài **thành phần liên thông** trên lưới. Duyệt mọi ô; gặp ô đất chưa thăm thì loang hết đảo, đếm số ô; **đánh dấu đã thăm** ngay khi thăm để mỗi ô xử lý đúng một lần.
5. Tránh các lỗi kinh điển: tính cả đường chéo, cộng mọi ô `1` thay vì từng đảo, quên đánh dấu đã thăm, không đặt lại bộ đếm cho mỗi đảo, so sánh với chuỗi `"1"`, đệ quy quá sâu.

## 2. Đề bài & ví dụ

Cho ma trận nhị phân `grid` kích thước `m x n`. Một **hòn đảo** là một nhóm các ô có giá trị `1` (đất) nối với nhau theo **4 hướng** (ngang hoặc dọc). Có thể xem **bốn cạnh của lưới được bao quanh bởi nước**. **Diện tích** hòn đảo là số ô `1` của nó.

Trả về **diện tích lớn nhất** của một hòn đảo trong `grid`. Nếu không có đảo nào, trả về `0`.

**Ví dụ 1:** lưới `8 x 13` (xem cell bên dưới) → đáp án `6`. Đáp án **không phải 11**: nếu tính cả đường chéo thì có một nhóm 11 ô, nhưng đề chỉ cho nối theo 4 hướng nên đảo lớn nhất chỉ có 6 ô.

**Ví dụ 2:** `grid = [[0,0,0,0,0,0,0,0]]` → `0` (không có đất).

**Ràng buộc:** `1 <= m, n <= 50`; mỗi ô là `0` hoặc `1`.

In [1]:
import sys
import time
import random
from collections import deque, Counter
from copy import deepcopy
from itertools import product
from typing import List

grid1 = [
    [0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0],
    [0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0],
    [0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0],
]
grid2 = [[0, 0, 0, 0, 0, 0, 0, 0]]


def show(grid, land="#", water="."):
    for row in grid:
        print("".join(land if v else water for v in row))


show(grid1)
print("m, n =", len(grid1), len(grid1[0]), "| số ô đất =", sum(map(sum, grid1)))

..#....#.....
.......###...
.##.#........
.#..##..#.#..
.#..##..###..
..........#..
.......###...
.......##....
m, n = 8 13 | số ô đất = 25


## 3. Kiến thức nền

### 3.1 Ma trận, chỉ số và việc sửa tại chỗ

`grid[i][j]` là ô hàng `i`, cột `j` (đếm từ 0); `len(grid)` là số hàng `m`, `len(grid[0])` là số cột `n`. Ở bài này mỗi ô là **số nguyên** `0` hoặc `1` (khác bài Dò mìn dùng ký tự). So sánh `grid[i][j] == 1` đúng, còn `grid[i][j] == "1"` **luôn sai**.

Nhiều lời giải đánh dấu "đã thăm" bằng cách **đổi ô `1` thành `0`** (sửa đầu vào). Tiện nhưng phá hủy dữ liệu, nên khi test ta `deepcopy` đầu vào để dùng lại.

In [2]:
g = [[1, 0], [1, 1]]
print(g[1][0] == 1, g[1][0] == "1", type(g[1][0]))
alias = g
alias[0][0] = 0
print("alias sửa -> g đổi theo:", g)
copy = deepcopy(g)
copy[1][1] = 0
print("deepcopy độc lập:", g, copy)

True False <class 'int'>
alias sửa -> g đổi theo: [[0, 0], [1, 1]]
deepcopy độc lập: [[0, 0], [1, 1]] [[0, 0], [1, 0]]


### 3.2 Bốn hướng lân cận và kiểm tra biên

Hàng xóm 4 hướng của `(i, j)`: `(i-1, j)`, `(i+1, j)`, `(i, j-1)`, `(i, j+1)`. Ta lưu `DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]`. Nhớ kiểm tra `0 <= x < m and 0 <= y < n` **trước** khi truy cập; Python cho phép chỉ số âm (`grid[-1]` là hàng cuối!) nên quên kiểm tra biên dưới sẽ **sai ngầm** mà không báo lỗi.

In [3]:
DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]


def neighbors4(i, j, m, n):
    return [(i + dx, j + dy) for dx, dy in DIRS if 0 <= i + dx < m and 0 <= j + dy < n]


print("giữa 3x3  :", neighbors4(1, 1, 3, 3))
print("góc (0,0) :", neighbors4(0, 0, 3, 3))
print("cạnh (0,1):", neighbors4(0, 1, 3, 3))
print("bẫy chỉ số âm: [[1],[2],[3]][-1] =", [[1], [2], [3]][-1])

giữa 3x3  : [(0, 1), (2, 1), (1, 0), (1, 2)]
góc (0,0) : [(1, 0), (0, 1)]
cạnh (0,1): [(1, 1), (0, 0), (0, 2)]
bẫy chỉ số âm: [[1],[2],[3]][-1] = [3]


### 3.3 Hàng đợi `deque`, ngăn xếp và đệ quy

- **Hàng đợi** (vào trước ra trước): `deque.append`, `popleft()` → duyệt theo lớp (**BFS**).
- **Ngăn xếp** (vào sau ra trước): `list.append`, `pop()` → duyệt sâu (**DFS** không đệ quy).
- **Đệ quy**: hàm tự gọi, mỗi lần chiếm một tầng *call stack*; mặc định Python chỉ cho khoảng **1000** tầng (`sys.getrecursionlimit()`), vượt thì `RecursionError`.

Cả ba đều thăm đúng các ô cùng một đảo; khác nhau ở thứ tự và cách quản lý danh sách chờ.

In [4]:
q = deque([1, 2, 3]); q.append(4)
print("queue:", q.popleft(), q.popleft(), list(q))
st = [1, 2, 3]; st.append(4)
print("stack:", st.pop(), st.pop(), st)
print("giới hạn đệ quy mặc định:", sys.getrecursionlimit())

queue: 1 2 [3, 4]
stack: 4 3 [1, 2]
giới hạn đệ quy mặc định: 1000


### 3.4 Đánh dấu đã thăm (visited)

Khi loang trên lưới, một ô có thể được nhiều ô khác "nhìn thấy". Nếu không đánh dấu thì nó bị xử lý lại, thậm chí **lặp vô hạn** (hai ô kề nhau cứ đẩy nhau). Có hai cách đánh dấu: (a) mảng `visited` riêng; (b) sửa chính `grid` (`1 → 0`). Quy tắc vàng: **đánh dấu ngay lúc quyết định thăm** (lúc đưa vào hàng đợi), không đợi lúc lấy ra.

### 3.5 Union-Find (DSU)

Cấu trúc **hợp các tập rời nhau**: `find(x)` trả đại diện của tập chứa `x` (có *nén đường đi*), `union(a, b)` gộp hai tập; ta thêm mảng `size` để biết kích thước tập. Dùng được khi ta nhìn bài toán là "gộp các ô đất kề nhau thành nhóm rồi lấy nhóm lớn nhất".

In [5]:
class DSU:
    def __init__(self, n):
        self.parent = list(range(n))
        self.size = [1] * n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]     # nén đường đi
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        if self.size[ra] < self.size[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra                                  # nối cây nhỏ vào cây lớn
        self.size[ra] += self.size[rb]
        return True


d = DSU(6)
d.union(0, 1); d.union(1, 2); d.union(4, 5)
print([d.size[d.find(i)] for i in range(6)], "(kích thước nhóm của từng phần tử)")

[3, 3, 3, 1, 2, 2] (kích thước nhóm của từng phần tử)


## 4. Ý tưởng nền tảng / trực giác

Nhìn lưới như bản đồ: "đất" là `#`, "nước" là `.`. Mỗi **hòn đảo** là một mảng đất liền. Muốn biết đảo to nhất, hãy **đi bộ**: đặt chân lên một ô đất chưa đi qua, rồi **đi khắp đảo**, đếm số ô đã bước, đồng thời **đánh dấu** mỗi ô đã bước để không đếm lại. Xong một đảo ta có diện tích của nó; tiếp tục tìm ô đất chưa đánh dấu kế tiếp. Lấy diện tích lớn nhất.

Ví dụ nhỏ (đã kiểm tra tay):

```
# # . . .
# . . # #
. . # # .
# . . . .
```

Đảo 1 gồm `(0,0), (0,1), (1,0)` — diện tích 3. Đảo 2 gồm `(1,3), (1,4), (2,3), (2,2)` — diện tích 4 (ô `(2,2)` nối `(2,3)` theo hàng ngang). Đảo 3 chỉ `(3,0)` — diện tích 1. Đáp án `4`. Chú ý: `(1,0)` và `(3,0)` **không** nối với nhau vì `(2,0)` là nước.

**Bẫy ví dụ 1:** lưới 8x13 có 25 ô đất chia thành nhiều đảo; đáp án chỉ `6`. Nếu tính nhầm cả đường chéo sẽ ra `11` (xem lỗi 1 ở mục 13).

In [6]:
small = [[1, 1, 0, 0, 0], [1, 0, 0, 1, 1], [0, 0, 1, 1, 0], [1, 0, 0, 0, 0]]
show(small)
print("tổng ô đất =", sum(map(sum, small)), "-> nhưng đáp án là diện tích của MỘT đảo")
print("tổng ô đất của ví dụ 1 =", sum(map(sum, grid1)), "nhưng đáp án = 6")

##...
#..##
..##.
#....
tổng ô đất = 8 -> nhưng đáp án là diện tích của MỘT đảo
tổng ô đất của ví dụ 1 = 25 nhưng đáp án = 6


## 5. Cách 1 — Brute Force: lan "nhãn nhỏ nhất" tới khi ổn định

**Ý tưởng không cần cấu trúc duyệt:** cho mỗi ô đất một **nhãn riêng** (ví dụ `i * n + j`). Sau đó lặp: với mỗi ô đất, nếu một hàng xóm đất có nhãn **nhỏ hơn**, lấy nhãn nhỏ đó. Lặp tới khi không còn ô nào đổi. Khi dừng, mọi ô cùng đảo có **cùng nhãn** (nhãn nhỏ nhất trong đảo) và ô khác đảo có nhãn khác. Đếm số ô mỗi nhãn bằng `Counter`, lấy lớn nhất.

Ý nghĩa: nhãn nhỏ nhất "lây" qua các ô liền kề như dịch bệnh; bao nhiêu vòng lặp là tuỳ độ dài đường đi trong đảo.

In [7]:
def max_area_labels(grid):
    m, n = len(grid), len(grid[0])
    label = {(i, j): i * n + j for i in range(m) for j in range(n) if grid[i][j] == 1}
    rounds = 0
    changed = True
    while changed:
        changed = False
        rounds += 1
        for (i, j) in label:
            best = label[(i, j)]
            for x, y in neighbors4(i, j, m, n):
                if (x, y) in label and label[(x, y)] < best:
                    best = label[(x, y)]
            if best < label[(i, j)]:
                label[(i, j)] = best
                changed = True
    max_area_labels.rounds = rounds
    return max(Counter(label.values()).values(), default=0)


print(max_area_labels(grid1), max_area_labels(grid2), max_area_labels(small))
print("số vòng lặp lan nhãn ở ví dụ 1:", max_area_labels.rounds)
assert max_area_labels(grid1) == 6 and max_area_labels(grid2) == 0 and max_area_labels(small) == 4

6 0 4
số vòng lặp lan nhãn ở ví dụ 1: 3


**Độ phức tạp:** mỗi vòng `O(m·n)`; số vòng tối đa cỡ độ dài đường đi dài nhất trong đảo, tới `O(m·n)` với đảo ngoằn ngoèo. Tổng xấu nhất `O((m·n)²)`; bộ nhớ `O(m·n)`. Đúng nhưng chậm.

**Nút thắt:** ta cứ quét lại toàn bộ lưới mỗi vòng, trong khi mỗi ô thực ra chỉ cần "gặp" đảo của mình **một lần**. Ý tưởng tốt hơn: đi từ một ô đất, **đi khắp đảo đúng một lần** rồi đánh dấu.

## 6. Ý tưởng thuật toán / bước đột phá

**Một đảo = một thành phần liên thông.** Thuật toán:

1. Duyệt mọi ô `(i, j)` theo thứ tự hàng, cột.
2. Nếu `grid[i][j] == 1` (đất **chưa thăm**): ta vừa tìm thấy một đảo mới. Loang từ ô này tới mọi ô đất liên thông, **đánh dấu từng ô** ngay khi thăm (đổi `1 → 0`) và đếm số ô.
3. Cập nhật `ans = max(ans, diện tích đảo vừa loang)`.

Vì mỗi ô đất bị đánh dấu ngay khi được thăm nên **mỗi ô xử lý đúng một lần**, tổng `O(m·n)`. "Loang" có thể làm bằng đệ quy (DFS), hàng đợi (BFS), ngăn xếp, hoặc coi như bài **gộp tập** (Union-Find).

## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta có thêm: **Cách 2** DFS đệ quy, **Cách 3** BFS bằng hàng đợi (kèm biến thể ngăn xếp), **Cách 4** Union-Find.

### Cách 2 — DFS đệ quy

**Giải thích từng đoạn:**

1. `dfs(i, j)` trả về **diện tích phần đảo còn chưa thăm** bắt đầu từ `(i, j)`.
2. Nếu `(i, j)` ngoài lưới hoặc là nước (`0`) → trả `0`.
3. Ngược lại đánh dấu đã thăm (`grid[i][j] = 0`) và trả `1 + dfs(4 hướng)`.
4. Vòng ngoài thử mọi ô và lấy `max`.

In [8]:
def max_area_dfs(grid):
    m, n = len(grid), len(grid[0])

    def dfs(i, j):
        if not (0 <= i < m and 0 <= j < n) or grid[i][j] == 0:
            return 0
        grid[i][j] = 0                       # đánh dấu đã thăm
        return 1 + dfs(i - 1, j) + dfs(i + 1, j) + dfs(i, j - 1) + dfs(i, j + 1)

    return max((dfs(i, j) for i in range(m) for j in range(n)), default=0)


print(max_area_dfs(deepcopy(grid1)), max_area_dfs(deepcopy(grid2)), max_area_dfs(deepcopy(small)))
assert max_area_dfs(deepcopy(grid1)) == 6 and max_area_dfs(deepcopy(grid2)) == 0 and max_area_dfs(deepcopy(small)) == 4

6 0 4


In [9]:
# Code nộp LeetCode cho Cách 2
class SolutionDFS:
    def maxAreaOfIsland(self, grid: List[List[int]]) -> int:
        sys.setrecursionlimit(10000)         # đảo ngoằn ngoèo 50x50 có thể sâu > 1000 tầng
        m, n = len(grid), len(grid[0])

        def dfs(i: int, j: int) -> int:
            if i < 0 or i >= m or j < 0 or j >= n or grid[i][j] == 0:
                return 0
            grid[i][j] = 0
            return 1 + dfs(i - 1, j) + dfs(i + 1, j) + dfs(i, j - 1) + dfs(i, j + 1)

        return max(dfs(i, j) for i in range(m) for j in range(n))

**Độ phức tạp:** thời gian `O(m·n)` (mỗi ô đất bị thăm một lần, mỗi lần xét 4 hướng); bộ nhớ `O(m·n)` cho call stack trong trường hợp xấu nhất (đảo hình rắn). Độ sâu đệ quy có thể vượt 1000, ta đo ở mục 12.

### Cách 3 — BFS bằng hàng đợi (an toàn, không đệ quy)

**Giải thích từng đoạn:**

1. Gặp ô đất `(i, j)`: đánh dấu thăm (`grid[i][j] = 0`), đặt `area = 1`, đưa vào `deque`.
2. Lấy một ô ra; với mỗi hàng xóm trong lưới **là đất**: đánh dấu thăm **ngay**, `area += 1`, đưa vào hàng đợi.
3. Khi hàng đợi rỗng, đảo đã loang xong; cập nhật `ans`.

Đổi `popleft()` thành `pop()` (và `deque` thành `list`) ta có **DFS dùng ngăn xếp**.

In [10]:
def max_area_bfs(grid):
    m, n = len(grid), len(grid[0])
    ans = 0
    for i in range(m):
        for j in range(n):
            if grid[i][j] == 1:
                grid[i][j] = 0
                area = 1
                queue = deque([(i, j)])
                while queue:
                    a, b = queue.popleft()
                    for x, y in neighbors4(a, b, m, n):
                        if grid[x][y] == 1:
                            grid[x][y] = 0           # đánh dấu NGAY khi đưa vào hàng đợi
                            area += 1
                            queue.append((x, y))
                ans = max(ans, area)
    return ans


def max_area_stack(grid):
    m, n = len(grid), len(grid[0])
    ans = 0
    for i in range(m):
        for j in range(n):
            if grid[i][j] == 1:
                grid[i][j] = 0
                area, stack = 1, [(i, j)]
                while stack:
                    a, b = stack.pop()               # LIFO thay cho FIFO
                    for x, y in neighbors4(a, b, m, n):
                        if grid[x][y] == 1:
                            grid[x][y] = 0
                            area += 1
                            stack.append((x, y))
                ans = max(ans, area)
    return ans


for fn in (max_area_bfs, max_area_stack):
    assert fn(deepcopy(grid1)) == 6 and fn(deepcopy(grid2)) == 0 and fn(deepcopy(small)) == 4
print("BFS và DFS-ngăn-xếp khớp 3 ví dụ")

BFS và DFS-ngăn-xếp khớp 3 ví dụ


In [11]:
# Code nộp LeetCode cho Cách 3 (BFS) - đây là đáp án chính
from collections import deque


class SolutionBFS:
    def maxAreaOfIsland(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        ans = 0
        for i in range(m):
            for j in range(n):
                if grid[i][j] == 1:
                    grid[i][j] = 0
                    area = 1
                    queue = deque([(i, j)])
                    while queue:
                        x, y = queue.popleft()
                        for dx, dy in ((-1, 0), (1, 0), (0, -1), (0, 1)):
                            a, b = x + dx, y + dy
                            if 0 <= a < m and 0 <= b < n and grid[a][b] == 1:
                                grid[a][b] = 0
                                area += 1
                                queue.append((a, b))
                    ans = max(ans, area)
        return ans

**Độ phức tạp:** thời gian `O(m·n)`; bộ nhớ `O(min(m·n, đảo lớn nhất))` cho hàng đợi — không có nguy cơ `RecursionError`.

### Cách 4 — Union-Find: gộp các ô đất kề nhau

**Ý tưởng khác:** đừng "đi" trên bản đồ; hãy **gộp tập**. Mỗi ô đất ban đầu là một tập riêng (kích thước 1). Duyệt mọi ô đất, với hàng xóm **bên phải** và **bên dưới** nếu cũng là đất thì `union`. (Chỉ cần hai hướng vì cặp `(a, b)` luôn được xét khi đứng ở ô phía trên/bên trái.) Sau cùng, đáp án là **kích thước tập lớn nhất** của các ô đất.

In [12]:
def max_area_dsu(grid):
    m, n = len(grid), len(grid[0])
    dsu = DSU(m * n)
    for i in range(m):
        for j in range(n):
            if grid[i][j] == 1:
                if i + 1 < m and grid[i + 1][j] == 1:
                    dsu.union(i * n + j, (i + 1) * n + j)
                if j + 1 < n and grid[i][j + 1] == 1:
                    dsu.union(i * n + j, i * n + j + 1)
    return max((dsu.size[dsu.find(i * n + j)] for i in range(m) for j in range(n) if grid[i][j] == 1), default=0)


print(max_area_dsu(grid1), max_area_dsu(grid2), max_area_dsu(small))
assert max_area_dsu(grid1) == 6 and max_area_dsu(grid2) == 0 and max_area_dsu(small) == 4

6 0 4


In [13]:
# Code nộp LeetCode cho Cách 4
class SolutionDSU:
    def maxAreaOfIsland(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        parent = list(range(m * n))
        size = [1] * (m * n)

        def find(x: int) -> int:
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        def union(a: int, b: int) -> None:
            ra, rb = find(a), find(b)
            if ra != rb:
                if size[ra] < size[rb]:
                    ra, rb = rb, ra
                parent[rb] = ra
                size[ra] += size[rb]

        for i in range(m):
            for j in range(n):
                if grid[i][j]:
                    if i + 1 < m and grid[i + 1][j]:
                        union(i * n + j, (i + 1) * n + j)
                    if j + 1 < n and grid[i][j + 1]:
                        union(i * n + j, i * n + j + 1)
        return max((size[find(i * n + j)] for i in range(m) for j in range(n) if grid[i][j]), default=0)

**Độ phức tạp:** `O(m·n·α(m·n))` ≈ `O(m·n)` thời gian; `O(m·n)` bộ nhớ (mảng `parent`, `size`). **Ưu điểm:** không sửa `grid`, không đệ quy, và rất hợp khi bài **thêm đất dần dần** (như "Number of Islands II").

## 8. TRACE từng bước

Lưới nhỏ `small` ở mục 4 (4 hàng, 5 cột). Bảng quét theo thứ tự hàng, cột:

| ô đang xét | giá trị | hành động | diện tích đảo vừa loang | `ans` sau |
|---|---|---|---|---|
| `(0,0)` | 1 | đảo mới, loang `(0,0),(0,1),(1,0)` | 3 | 3 |
| `(0,1)` | 0 (đã thăm) | bỏ qua | — | 3 |
| `(1,0)` | 0 (đã thăm) | bỏ qua | — | 3 |
| `(1,3)` | 1 | đảo mới, loang `(1,3),(1,4),(2,3),(2,2)` | 4 | 4 |
| `(3,0)` | 1 | đảo mới, chỉ `(3,0)` | 1 | 4 |
| các ô còn lại | 0 | bỏ qua | — | 4 |

Hàm trace in từng lần lấy ô ra khỏi hàng đợi của BFS và bảng sau khi loang xong mỗi đảo (ô đã thăm hiển thị là `.`):

In [14]:
def trace_bfs(grid):
    grid = deepcopy(grid)
    m, n = len(grid), len(grid[0])
    ans, island = 0, 0
    for i in range(m):
        for j in range(n):
            if grid[i][j] != 1:
                continue
            island += 1
            grid[i][j] = 0
            area = 1
            queue = deque([(i, j)])
            print(f"--- đảo #{island}: xuất phát tại {(i, j)}")
            while queue:
                a, b = queue.popleft()
                added = []
                for x, y in neighbors4(a, b, m, n):
                    if grid[x][y] == 1:
                        grid[x][y] = 0
                        area += 1
                        queue.append((x, y))
                        added.append((x, y))
                print(f"    lấy {(a, b)} -> thêm {added if added else '-'} | area={area} | hàng đợi={list(queue)}")
            ans = max(ans, area)
            print(f"    diện tích đảo #{island} = {area}, ans = {ans}")
            show(grid)
    print("=> đáp án", ans)
    return ans


assert trace_bfs(small) == 4

--- đảo #1: xuất phát tại (0, 0)
    lấy (0, 0) -> thêm [(1, 0), (0, 1)] | area=3 | hàng đợi=[(1, 0), (0, 1)]
    lấy (1, 0) -> thêm - | area=3 | hàng đợi=[(0, 1)]
    lấy (0, 1) -> thêm - | area=3 | hàng đợi=[]
    diện tích đảo #1 = 3, ans = 3
.....
...##
..##.
#....
--- đảo #2: xuất phát tại (1, 3)
    lấy (1, 3) -> thêm [(2, 3), (1, 4)] | area=3 | hàng đợi=[(2, 3), (1, 4)]
    lấy (2, 3) -> thêm [(2, 2)] | area=4 | hàng đợi=[(1, 4), (2, 2)]
    lấy (1, 4) -> thêm - | area=4 | hàng đợi=[(2, 2)]
    lấy (2, 2) -> thêm - | area=4 | hàng đợi=[]
    diện tích đảo #2 = 4, ans = 4
.....
.....
.....
#....
--- đảo #3: xuất phát tại (3, 0)
    lấy (3, 0) -> thêm - | area=1 | hàng đợi=[]
    diện tích đảo #3 = 1, ans = 4
.....
.....
.....
.....
=> đáp án 4


**Trace DFS đệ quy** (thụt lề theo độ sâu) cho cùng lưới, chỉ in các ô được thăm:

In [15]:
def trace_dfs(grid):
    grid = deepcopy(grid)
    m, n = len(grid), len(grid[0])

    def dfs(i, j, depth):
        if not (0 <= i < m and 0 <= j < n) or grid[i][j] == 0:
            return 0
        grid[i][j] = 0
        print("  " * depth + f"thăm {(i, j)}")
        total = 1
        for dx, dy in DIRS:
            total += dfs(i + dx, j + dy, depth + 1)
        return total

    ans = 0
    for i in range(m):
        for j in range(n):
            if grid[i][j] == 1:
                print(f"đảo mới tại {(i, j)}")
                area = dfs(i, j, 0)
                print(f"=> diện tích {area}")
                ans = max(ans, area)
    return ans


assert trace_dfs(small) == 4

đảo mới tại (0, 0)
thăm (0, 0)
  thăm (1, 0)
  thăm (0, 1)
=> diện tích 3
đảo mới tại (1, 3)
thăm (1, 3)
  thăm (2, 3)
    thăm (2, 2)
  thăm (1, 4)
=> diện tích 4
đảo mới tại (3, 0)
thăm (3, 0)
=> diện tích 1


## 9. Code nộp LeetCode

Chọn **BFS**: `O(m·n)` thời gian, không phụ thuộc giới hạn đệ quy. Giữ nguyên `class Solution`, `maxAreaOfIsland(self, grid)`:

In [16]:
from collections import deque
from typing import List


class Solution:
    def maxAreaOfIsland(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        ans = 0
        for i in range(m):
            for j in range(n):
                if grid[i][j] == 1:              # gặp đất chưa thăm: một đảo mới
                    grid[i][j] = 0               # đánh dấu đã thăm
                    area = 1
                    queue = deque([(i, j)])
                    while queue:
                        x, y = queue.popleft()
                        for dx, dy in ((-1, 0), (1, 0), (0, -1), (0, 1)):
                            a, b = x + dx, y + dy
                            if 0 <= a < m and 0 <= b < n and grid[a][b] == 1:
                                grid[a][b] = 0   # đánh dấu NGAY khi đưa vào hàng đợi
                                area += 1
                                queue.append((a, b))
                    ans = max(ans, area)
        return ans


print(Solution().maxAreaOfIsland(deepcopy(grid1)), Solution().maxAreaOfIsland(deepcopy(grid2)))
assert Solution().maxAreaOfIsland(deepcopy(grid1)) == 6 and Solution().maxAreaOfIsland(deepcopy(grid2)) == 0

6 0


## 10. Vì sao đúng

**Bất biến:** tại mọi thời điểm, các ô đã bị đổi thành `0` là các ô đất **đã được tính** vào diện tích của đúng một đảo (đảo đang hoặc đã loang xong).

1. *Đủ:* vòng ngoài thử mọi ô; mỗi đảo có ít nhất một ô, và ô đầu tiên của đảo (theo thứ tự quét) sẽ được gặp khi còn là `1` — vì chỉ các ô thuộc đảo **khác** hoặc đã loang xong mới bị đổi, và đảo này chưa loang. Nên mọi đảo được loang đúng một lần.
2. *Chuẩn:* khi loang từ một ô, hàng đợi chứa các ô "đã tính nhưng chưa mở rộng"; ta mở rộng mọi hàng xóm đất chưa thăm. Khi hàng đợi rỗng, mọi ô đất liên thông với ô xuất phát (theo 4 hướng) đều đã tính: vì nếu có ô đất liên thông chưa tính thì tồn tại một ô đã tính kề một ô chưa tính, mà ô đã tính đó đã được mở rộng, mâu thuẫn.
3. *Không đếm lại:* đánh dấu ngay khi đưa vào hàng đợi nên mỗi ô vào hàng đợi tối đa một lần; do đó `area` đúng bằng số ô của đảo, và tổng thời gian `O(m·n)`.
4. *Đáp án:* `max` diện tích các đảo; không có đảo → `ans = 0`.

Ta kiểm chứng bằng máy: 4 cách khác nhau về ý tưởng cho cùng đáp án trên **mọi** lưới `3×3` và `3×4` (4608 lưới) và hàng nghìn lưới ngẫu nhiên (mục 12).

## 11. Time / Space Complexity

Đặt `N = m·n`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Lan nhãn nhỏ nhất | `O(N²)` xấu nhất | `O(N)` | chỉ làm oracle |
| 2. DFS đệ quy | `O(N)` | `O(N)` call stack | nâng `setrecursionlimit` khi đảo rắn |
| 3. BFS / DFS ngăn xếp | `O(N)` | `O(N)` heap | **đáp án chính** |
| 4. Union-Find | `O(N·α(N))` | `O(N)` | không sửa `grid` |

Đo thực tế trên lưới `50×50` (tối đa theo đề):

In [17]:
def random_grid(rng, m, n, p):
    """Lưới m x n; mỗi ô là đất (1) với xác suất p."""
    return [[1 if rng.random() < p else 0 for _ in range(n)] for _ in range(m)]


def timeit(fn, grid, repeat=5):
    best = 1e9
    for _ in range(repeat):
        g = deepcopy(grid)
        t0 = time.perf_counter()
        fn(g)
        best = min(best, time.perf_counter() - t0)
    return best


rng = random.Random(2024)
old = sys.getrecursionlimit()
sys.setrecursionlimit(20000)
for p in (0.3, 0.6, 1.0):
    g = random_grid(rng, 50, 50, p)
    print(f"lưới 50x50, mật độ đất {p}: đáp án = {max_area_bfs(deepcopy(g))}")
    for name, fn in [("labels", max_area_labels), ("dfs", max_area_dfs), ("bfs", max_area_bfs),
                     ("stack", max_area_stack), ("dsu", max_area_dsu)]:
        print(f"   {name:7s} {timeit(fn, g, 3 if name == 'labels' else 5) * 1000:8.2f} ms")
sys.setrecursionlimit(old)

lưới 50x50, mật độ đất 0.3: đáp án = 12
   labels      4.59 ms
   dfs         0.78 ms
   bfs         0.88 ms
   stack       0.79 ms
   dsu         0.72 ms
lưới 50x50, mật độ đất 0.6: đáp án = 1093


   labels     70.64 ms
   dfs         1.15 ms
   bfs         1.35 ms
   stack       1.18 ms
   dsu         1.29 ms
lưới 50x50, mật độ đất 1.0: đáp án = 2500
   labels      5.90 ms
   dfs         1.83 ms
   bfs         2.12 ms
   stack       1.88 ms
   dsu         2.50 ms


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ: cài đặt sai phải bị bắt, (D) vét cạn mọi lưới nhỏ, (E) ngẫu nhiên có seed, (F) giới hạn lớn nhất và đệ quy.

### A. Ví dụ của đề

In [18]:
IMPLS = {
    "labels": max_area_labels,
    "dfs": max_area_dfs,
    "bfs": max_area_bfs,
    "stack": max_area_stack,
    "dsu": max_area_dsu,
    "Solution": lambda g: Solution().maxAreaOfIsland(g),
    "SolutionDFS": lambda g: SolutionDFS().maxAreaOfIsland(g),
    "SolutionBFS": lambda g: SolutionBFS().maxAreaOfIsland(g),
    "SolutionDSU": lambda g: SolutionDSU().maxAreaOfIsland(g),
}


def check(grid, want):
    for name, fn in IMPLS.items():
        got = fn(deepcopy(grid))
        assert got == want, (name, got, want)


check(grid1, 6)
check(grid2, 0)
print("ví dụ 1, 2: cả", len(IMPLS), "cài đặt đúng")

ví dụ 1, 2: cả 9 cài đặt đúng


### B. Edge case

- `1×1`: đất → `1`, nước → `0`.
- Một hàng, một cột.
- Toàn đất `50×50` → `2500`.
- **Bàn cờ** (đất xen nước): các ô đất chỉ chạm chéo nên mỗi đảo có đúng 1 ô → `1` (kiểm tra "không tính chéo").
- Ô đất nằm sát biên/góc.
- **Rắn ngoằn ngoèo**: một đảo duy nhất nhưng đường đi rất dài.
- Một **vòng** đất bao quanh một ô nước.

In [19]:
def checkerboard(m, n):
    return [[(i + j) % 2 for j in range(n)] for i in range(m)]


def serpentine(m, n):
    """Một đảo hình rắn: hàng chẵn toàn đất, hàng lẻ chỉ có một ô đất ở đầu xen kẽ phải/trái."""
    g = [[0] * n for _ in range(m)]
    for i in range(0, m, 2):
        g[i] = [1] * n
    for i in range(1, m, 2):
        g[i][n - 1 if (i // 2) % 2 == 0 else 0] = 1
    return g


ring = [[1, 1, 1], [1, 0, 1], [1, 1, 1]]
cases = [
    ([[1]], 1), ([[0]], 0),
    ([[1, 1, 0, 1, 1, 1]], 3), ([[1], [1], [0], [1]], 2),
    ([[1] * 50 for _ in range(50)], 2500),
    (checkerboard(5, 5), 1), (checkerboard(1, 7), 1),
    ([[1, 0], [0, 1]], 1),
    (ring, 8), ([[0, 0, 0], [0, 1, 0], [0, 0, 0]], 1),
    ([[1, 0, 0], [0, 0, 0], [0, 0, 1]], 1),
]
for g, want in cases:
    check(g, want)
sp = serpentine(7, 5)
show(sp)
check(sp, sum(map(sum, sp)))
print(len(cases) + 1, "edge case: tất cả cài đặt đúng; rắn 7x5 là một đảo", sum(map(sum, sp)), "ô")

#####
....#
#####
#....
#####
....#
#####
12 edge case: tất cả cài đặt đúng; rắn 7x5 là một đảo 23 ô


### C. Phản ví dụ: cài đặt sai phải bị bắt

Ta viết các bản sai (chi tiết ở mục 13) và chắc chắn có input làm chúng khác đáp án đúng:

In [20]:
def bug_eight_dirs(grid):                    # tính cả đường chéo
    m, n = len(grid), len(grid[0])
    d8 = [(dx, dy) for dx in (-1, 0, 1) for dy in (-1, 0, 1) if (dx, dy) != (0, 0)]
    ans = 0
    for i in range(m):
        for j in range(n):
            if grid[i][j] == 1:
                grid[i][j] = 0
                area, q = 1, deque([(i, j)])
                while q:
                    a, b = q.popleft()
                    for dx, dy in d8:
                        x, y = a + dx, b + dy
                        if 0 <= x < m and 0 <= y < n and grid[x][y] == 1:
                            grid[x][y] = 0
                            area += 1
                            q.append((x, y))
                ans = max(ans, area)
    return ans


def bug_sum_all(grid):                       # cộng mọi ô đất
    return sum(map(sum, grid))


def bug_no_reset(grid):                      # không đặt lại bộ đếm cho mỗi đảo
    m, n = len(grid), len(grid[0])
    ans = area = 0
    for i in range(m):
        for j in range(n):
            if grid[i][j] == 1:
                grid[i][j] = 0
                area += 1                    # SAI: area không về 0 giữa các đảo
                q = deque([(i, j)])
                while q:
                    a, b = q.popleft()
                    for x, y in neighbors4(a, b, m, n):
                        if grid[x][y] == 1:
                            grid[x][y] = 0
                            area += 1
                            q.append((x, y))
                ans = max(ans, area)
    return ans


def bug_negative_index(grid):                # quên kiểm tra biên dưới: x >= 0, y >= 0
    m, n = len(grid), len(grid[0])

    def dfs(i, j):
        if i >= m or j >= n or grid[i][j] == 0:
            return 0
        grid[i][j] = 0
        return 1 + dfs(i - 1, j) + dfs(i + 1, j) + dfs(i, j - 1) + dfs(i, j + 1)

    old = sys.getrecursionlimit()
    sys.setrecursionlimit(20000)
    try:
        return max(dfs(i, j) for i in range(m) for j in range(n))
    finally:
        sys.setrecursionlimit(old)


def bug_string_compare(grid):                # so với "1" thay vì 1
    m, n = len(grid), len(grid[0])
    ans = 0
    for i in range(m):
        for j in range(n):
            if grid[i][j] == "1":
                ans = 1
    return ans


BUGS = [bug_eight_dirs, bug_sum_all, bug_no_reset, bug_negative_index, bug_string_compare]
rng = random.Random(1)
tests = [random_grid(rng, rng.randint(1, 6), rng.randint(1, 6), rng.choice([0.3, 0.5, 0.7])) for _ in range(600)]
for bug in BUGS:
    bad = None
    for t in tests:
        try:
            got = bug(deepcopy(t))
        except Exception as e:
            got = type(e).__name__
        if got != max_area_bfs(deepcopy(t)):
            bad = (t, got)
            break
    assert bad is not None, bug.__name__
    print(f"{bug.__name__:22s} sai trên lưới {bad[0]} -> {bad[1]} (đúng {max_area_bfs(deepcopy(bad[0]))})")

bug_eight_dirs         sai trên lưới [[0, 0, 1, 0, 1], [1, 0, 0, 1, 1], [1, 0, 1, 1, 0], [0, 0, 1, 0, 0], [1, 0, 0, 0, 0], [0, 0, 0, 0, 0]] -> 7 (đúng 6)
bug_sum_all            sai trên lưới [[1, 0, 0, 1, 1, 1]] -> 4 (đúng 3)
bug_no_reset           sai trên lưới [[1, 0, 0, 1, 1, 1]] -> 4 (đúng 3)
bug_negative_index     sai trên lưới [[1, 0, 0, 1, 1, 1]] -> 4 (đúng 3)
bug_string_compare     sai trên lưới [[1, 0, 0, 0, 0], [1, 1, 0, 0, 0]] -> 0 (đúng 3)


### D. Vét cạn mọi lưới nhỏ

Liệt kê **mọi** lưới nhị phân `3×3` (`2⁹ = 512`) và `3×4`, `4×3` (`2¹² = 4096` mỗi loại, tổng `8704` lưới) — gồm cả mọi hình dạng đảo có thể có ở cỡ đó. Mọi cài đặt phải khớp **oracle lan nhãn**, và oracle lan nhãn cũng được kiểm với cách tính độc lập bằng tập hợp (`frozenset` các ô của từng đảo).

In [21]:
def islands_by_sets(grid):
    """Oracle thứ hai: tách đảo bằng tập hợp, hoàn toàn độc lập với các cách trên."""
    m, n = len(grid), len(grid[0])
    lands = {(i, j) for i in range(m) for j in range(n) if grid[i][j] == 1}
    areas = []
    while lands:
        comp = {lands.pop()}
        grew = True
        while grew:
            grew = False
            for (i, j) in list(comp):
                for x, y in neighbors4(i, j, m, n):
                    if (x, y) in lands:
                        lands.remove((x, y))
                        comp.add((x, y))
                        grew = True
        areas.append(len(comp))
    return max(areas, default=0)


total = 0
for m, n in ((3, 3), (3, 4), (4, 3)):
    for bits in product((0, 1), repeat=m * n):
        g = [list(bits[r * n:(r + 1) * n]) for r in range(m)]
        want = islands_by_sets(g)
        assert max_area_labels(deepcopy(g)) == want
        for name, fn in IMPLS.items():
            assert fn(deepcopy(g)) == want, (name, g)
        total += 1
print(total, "lưới vét cạn: 9 cài đặt đều khớp oracle độc lập")

8704 lưới vét cạn: 9 cài đặt đều khớp oracle độc lập


### E. Ngẫu nhiên có seed, đúng ràng buộc

In [22]:
rng = random.Random(20240695)
old = sys.getrecursionlimit()
sys.setrecursionlimit(20000)
n_tests = 0
for trial in range(1500):
    m, n = rng.randint(1, 12), rng.randint(1, 12)
    g = random_grid(rng, m, n, rng.choice([0.1, 0.3, 0.5, 0.6, 0.8]))
    want = islands_by_sets(g)
    for name, fn in IMPLS.items():
        assert fn(deepcopy(g)) == want, (name, g)
    n_tests += 1
for trial in range(60):                       # lưới lớn, đúng ràng buộc 50x50
    m, n = rng.randint(30, 50), rng.randint(30, 50)
    g = random_grid(rng, m, n, rng.choice([0.4, 0.55, 0.7]))
    want = islands_by_sets(g)
    for name, fn in IMPLS.items():
        if name == "labels":
            continue
        assert fn(deepcopy(g)) == want, (name,)
    n_tests += 1
sys.setrecursionlimit(old)
print(n_tests, "lưới ngẫu nhiên (cả 50x50): mọi cài đặt trùng oracle")

# tính chất: thêm 1 ô đất vào ô nước không làm đáp án giảm và không vượt tổng số ô đất
rng = random.Random(5)
for _ in range(300):
    g = random_grid(rng, 6, 6, 0.4)
    zeros = [(i, j) for i in range(6) for j in range(6) if g[i][j] == 0]
    if not zeros:
        continue
    i, j = rng.choice(zeros)
    g2 = deepcopy(g)
    g2[i][j] = 1
    a, b = Solution().maxAreaOfIsland(deepcopy(g)), Solution().maxAreaOfIsland(deepcopy(g2))
    assert a <= b <= sum(map(sum, g2))
print("tính chất đơn điệu (thêm đất không làm đáp án giảm): OK")

1560 lưới ngẫu nhiên (cả 50x50): mọi cài đặt trùng oracle
tính chất đơn điệu (thêm đất không làm đáp án giảm): OK


### F. Giới hạn lớn nhất và đệ quy

Đảo hình rắn `50×50` là ca xấu nhất cho DFS đệ quy. Ta đo độ sâu và chứng kiến `RecursionError` với giới hạn mặc định 1000.

In [23]:
snake = serpentine(50, 50)
land = sum(map(sum, snake))
print("rắn 50x50 có", land, "ô đất, một đảo duy nhất")
for name, fn in IMPLS.items():
    old = sys.getrecursionlimit()
    got = fn(deepcopy(snake))
    sys.setrecursionlimit(old)
    assert got == land, name

# DFS thuần (không nâng giới hạn) với giới hạn 1000
old = sys.getrecursionlimit()
sys.setrecursionlimit(1000)
try:
    max_area_dfs(deepcopy(snake))
    print("không lỗi (bất ngờ)")
except RecursionError:
    print("max_area_dfs + giới hạn 1000 -> RecursionError (đảo rắn quá sâu)")
finally:
    sys.setrecursionlimit(old)

full = [[1] * 50 for _ in range(50)]
for name in ("bfs", "stack", "dsu", "Solution", "SolutionBFS", "SolutionDSU"):
    assert IMPLS[name](deepcopy(full)) == 2500
print("toàn đất 50x50 -> 2500: BFS, ngăn xếp, DSU đều đúng")

rắn 50x50 có 1275 ô đất, một đảo duy nhất


max_area_dfs + giới hạn 1000 -> RecursionError (đảo rắn quá sâu)
toàn đất 50x50 -> 2500: BFS, ngăn xếp, DSU đều đúng


## 13. Các lỗi người mới hay gặp

### Lỗi 1: tính cả đường chéo (8 hướng)

Đề nói rõ "connected **4-directionally**". Bàn cờ `[[1,0],[0,1]]` có hai đảo, mỗi đảo 1 ô; nếu tính chéo ra `2`.

In [24]:
g = [[1, 0], [0, 1]]
print("bug_eight_dirs:", bug_eight_dirs(deepcopy(g)), "| đúng:", Solution().maxAreaOfIsland(deepcopy(g)))
g = checkerboard(3, 3)
print("bàn cờ 3x3    : sai", bug_eight_dirs(deepcopy(g)), "| đúng", Solution().maxAreaOfIsland(deepcopy(g)))
print("ví dụ 1 với 8 hướng:", bug_eight_dirs(deepcopy(grid1)), "(chính là con số 11 mà đề nhắc tới)")
assert bug_eight_dirs([[1, 0], [0, 1]]) == 2 and bug_eight_dirs(deepcopy(grid1)) == 11

bug_eight_dirs: 2 | đúng: 1
bàn cờ 3x3    : sai 4 | đúng 1
ví dụ 1 với 8 hướng: 11 (chính là con số 11 mà đề nhắc tới)


**Vì sao sai:** hai ô chỉ chạm góc không nối với nhau. **Đúng:** `DIRS` chỉ 4 hướng.

### Lỗi 2: cộng mọi ô đất

Nhầm "diện tích lớn nhất của một đảo" với "tổng đất". Ví dụ 1 có 25 ô đất nhưng đáp án là `6`.

In [25]:
print("bug_sum_all(ví dụ 1) =", bug_sum_all(deepcopy(grid1)), "| đúng =", Solution().maxAreaOfIsland(deepcopy(grid1)))
assert bug_sum_all(deepcopy(grid1)) == 25

bug_sum_all(ví dụ 1) = 25 | đúng = 6


**Vì sao sai:** phải đếm **theo từng đảo** rồi lấy `max`. **Đúng:** khởi tạo `area = 1` (hoặc `0`) **mỗi khi bắt đầu đảo mới**.

### Lỗi 3: quên đặt lại bộ đếm cho đảo mới

Khai báo `area` ngoài vòng lặp rồi cộng dồn: kết quả là tổng các đảo đã duyệt, không phải đảo lớn nhất.

In [26]:
g = [[1, 0, 1], [0, 0, 0], [1, 0, 1]]
print("bug_no_reset =", bug_no_reset(deepcopy(g)), "| đúng =", Solution().maxAreaOfIsland(deepcopy(g)))
assert bug_no_reset(deepcopy(g)) == 4 and Solution().maxAreaOfIsland(deepcopy(g)) == 1

bug_no_reset = 4 | đúng = 1


**Vì sao sai:** 4 đảo 1 ô nhưng `area` cộng dồn thành `4`. **Đúng:** `area = 1` ngay trong nhánh `if grid[i][j] == 1`.

### Lỗi 4: quên đánh dấu đã thăm → lặp vô hạn

Nếu chỉ kiểm tra "là đất" mà không đổi `1 → 0` (hay `visited`), hai ô kề nhau liên tục đẩy nhau vào hàng đợi. Dưới đây có bộ đếm bảo vệ để không treo notebook:

In [27]:
def bug_no_visited(grid, guard=5000):
    m, n = len(grid), len(grid[0])
    queue = deque([(0, 0)])
    pushes = 0
    area = 0
    while queue:
        a, b = queue.popleft()
        area += 1
        for x, y in neighbors4(a, b, m, n):
            if grid[x][y] == 1:               # SAI: không đánh dấu đã thăm
                queue.append((x, y))
                pushes += 1
                if pushes > guard:
                    raise RuntimeError(f"vòng lặp không kết thúc: đã đẩy {pushes} lần trong lưới chỉ {m * n} ô")
    return area


try:
    bug_no_visited([[1, 1], [1, 1]])
except RuntimeError as e:
    print("CODE SAI ->", e)
print("CODE ĐÚNG -> diện tích", Solution().maxAreaOfIsland([[1, 1], [1, 1]]))

CODE SAI -> vòng lặp không kết thúc: đã đẩy 5001 lần trong lưới chỉ 4 ô
CODE ĐÚNG -> diện tích 4


**Vì sao sai:** không có gì ngăn quay lại. **Đúng:** đổi `grid[x][y] = 0` (hoặc ghi vào `visited`) **ngay khi đưa vào hàng đợi**.

### Lỗi 5: so sánh với chuỗi `"1"` hoặc quên biên dưới

Ở bài này ô là **số nguyên**. `grid[i][j] == "1"` luôn `False` → luôn trả `0`. Quên `x >= 0` thì `grid[-1]` "quấn vòng" sang hàng cuối, nối các ô ở hai mép đối diện thành một đảo.

In [28]:
print("so sánh với '1':", bug_string_compare(deepcopy(grid1)), "| đúng:", Solution().maxAreaOfIsland(deepcopy(grid1)), "(bug_string_compare trả 0)")
g = [[1, 0, 0], [0, 0, 0], [1, 0, 0]]            # hai ô cách nhau 2 hàng; hàng -1 là hàng cuối
print("thiếu biên dưới:", bug_negative_index(deepcopy(g)), "| đúng:", Solution().maxAreaOfIsland(deepcopy(g)))
assert bug_negative_index(deepcopy(g)) == 2 and Solution().maxAreaOfIsland(deepcopy(g)) == 1

so sánh với '1': 0 | đúng: 6 (bug_string_compare trả 0)
thiếu biên dưới: 2 | đúng: 1


**Vì sao sai:** `(0,0)` có hàng xóm `(-1,0)` được Python hiểu là `(2,0)` nên hai ô đất ở hai mép nối với nhau. **Đúng:** luôn viết đủ `0 <= x < m and 0 <= y < n`.

### Lỗi 6 (bonus): quên rằng hàm sửa đầu vào

Cách đánh dấu `1 → 0` phá `grid`. Nếu bạn test hai lần trên cùng một biến, lần hai sẽ thấy lưới toàn `0` và ra `0`. Luôn `deepcopy` khi test.

In [29]:
g = deepcopy(grid1)
print("lần 1:", Solution().maxAreaOfIsland(g), "| lần 2 trên cùng biến:", Solution().maxAreaOfIsland(g), "(lưới đã bị xoá sạch)")

lần 1: 6 | lần 2 trên cùng biến: 0 (lưới đã bị xoá sạch)


## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** đề hỏi về "nhóm ô liên thông" (đảo, vùng, hồ, bộ phận) trên lưới → **flood fill** bằng DFS/BFS hoặc Union-Find.

**Khung chung:**

```
for mỗi ô:
    nếu ô thoả điều kiện và chưa thăm:
        loang từ ô này (đánh dấu ngay khi thăm), tính đại lượng cần
        cập nhật đáp án
```

**Mẹo:**

1. Đọc kỹ **số hướng** (4 hay 8) và **kiểu ô** (số hay chuỗi).
2. Đánh dấu **ngay** khi đưa vào danh sách chờ.
3. Đệ quy sâu tới `N` → ưu tiên BFS/ngăn xếp khi `N` lớn (ở đây `N = 2500`).
4. Muốn giữ `grid` nguyên: dùng `visited` hoặc Union-Find.
5. Khi cần kích thước **từng** thành phần: lấy kết quả của hàm loang (trả về số ô) hoặc `size[]` của DSU.

**Họ bài cùng pattern:** Number of Islands (200), Flood Fill (733), Island Perimeter (463), Surrounded Regions (130), Number of Enclaves (1020), Making A Large Island (827), Minesweeper (529).

## 15. Biến thể

1. **Đếm số đảo** (200): đếm số lần "gặp ô đất mới" ở vòng ngoài.
2. **Chu vi đảo** (463): mỗi ô đất đóng góp `4 − (số hàng xóm đất)`.
3. **Cho lật đúng một ô nước thành đất** (827): tìm đảo lớn nhất có thể (bài tập 5).
4. **Liệt kê diện tích mọi đảo** theo thứ tự giảm dần.
5. **8 hướng:** chỉ cần thay `DIRS`.

Cài đặt biến thể 1, 2, 4 và đối chiếu:

In [30]:
def island_areas(grid):
    """Biến thể 4: diện tích mọi đảo, sắp giảm dần (và biến thể 1: số đảo = len)."""
    grid = deepcopy(grid)
    m, n = len(grid), len(grid[0])
    areas = []
    for i in range(m):
        for j in range(n):
            if grid[i][j] == 1:
                grid[i][j] = 0
                area, q = 1, deque([(i, j)])
                while q:
                    a, b = q.popleft()
                    for x, y in neighbors4(a, b, m, n):
                        if grid[x][y] == 1:
                            grid[x][y] = 0
                            area += 1
                            q.append((x, y))
                areas.append(area)
    return sorted(areas, reverse=True)


def perimeter(grid):
    """Biến thể 2: chu vi tổng của mọi đảo (không tính 'hồ' bên trong khác mép lưới)."""
    m, n = len(grid), len(grid[0])
    total = 0
    for i in range(m):
        for j in range(n):
            if grid[i][j] == 1:
                total += 4 - sum(1 for x, y in neighbors4(i, j, m, n) if grid[x][y] == 1)
    return total


print(island_areas(grid1), "-> số đảo", len(island_areas(grid1)))
print(island_areas(small), "| chu vi tổng", perimeter(small))
assert island_areas(grid1)[0] == 6 and island_areas(small) == [4, 3, 1]
assert perimeter([[1]]) == 4 and perimeter([[1, 1]]) == 6 and perimeter([[1, 1], [1, 1]]) == 8

[6, 5, 5, 4, 4, 1] -> số đảo 6
[4, 3, 1] | chu vi tổng 22


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Lan nhãn nhỏ nhất | Mỗi ô đất một nhãn, lan nhãn nhỏ nhất tới ổn định | `O(N²)` | `O(N)` | Làm oracle để kiểm chứng |
| 2. DFS đệ quy | Loang đệ quy, trả về diện tích | `O(N)` | `O(N)` call stack | Code ngắn; nhớ nâng `setrecursionlimit` |
| 3. BFS / DFS ngăn xếp | Hàng đợi/ngăn xếp, đánh dấu khi đưa vào | `O(N)` | `O(N)` heap | **Đáp án chính**, an toàn |
| 4. Union-Find | Gộp ô đất kề nhau, lấy `size` lớn nhất | `O(N·α(N))` | `O(N)` | Không sửa `grid`; đất thêm dần |

Cả bốn cho cùng đáp án trên 8704 lưới vét cạn và 1560 lưới ngẫu nhiên (mục 12).

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi mới chạy:

In [31]:
g = [[1, 1, 0], [0, 1, 0], [1, 0, 1]]
print(len(g), len(g[0]), sum(map(sum, g)))
print(max_area_bfs(deepcopy(g)))
print(island_areas(g))

3 3 5
3
[3, 1, 1]


### Bài 2 (tính tay)

Lưới `4 × 5`:

```
1 1 0 1 1
1 0 0 1 0
0 0 1 1 0
1 0 0 0 1
```

- (a) Liệt kê các đảo (toạ độ các ô) và diện tích.
- (b) Đáp án bài `maxAreaOfIsland`?
- (c) Nếu tính cả đường chéo (8 hướng), đáp án là bao nhiêu và vì sao khác?

Tự giải rồi kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện hàm `dfs` trả về diện tích phần đảo bắt đầu tại `(i, j)`:

In [32]:
def max_area_todo(grid):
    m, n = len(grid), len(grid[0])

    def dfs(i, j):
        # TODO 1: nếu (i, j) ngoài lưới hoặc là nước -> trả 0
        # TODO 2: đánh dấu đã thăm
        # TODO 3: trả 1 + diện tích 4 hướng
        pass

    return max(dfs(i, j) for i in range(m) for j in range(n))


# Khi làm xong, bỏ comment để kiểm tra:
# sys.setrecursionlimit(10000)
# assert max_area_todo(deepcopy(grid1)) == 6
# assert max_area_todo([[0, 0]]) == 0

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết lớp `MySolution` bằng BFS (hoặc DSU):

In [33]:
class MySolution:
    def maxAreaOfIsland(self, grid):
        pass            # TODO


# Khi làm xong:
# for g, w in [(grid1, 6), (grid2, 0), (small, 4)]:
#     assert MySolution().maxAreaOfIsland(deepcopy(g)) == w

### Bài 5 (mini challenge): được lật một ô nước

Cho phép đổi **tối đa một** ô `0` thành `1`. Viết `largest_after_flip(grid)` trả về diện tích đảo lớn nhất có thể (nếu lưới toàn đất thì là `m·n`; nếu toàn nước thì `1`). Gợi ý: gán nhãn mỗi đảo kèm diện tích; với mỗi ô nước, cộng diện tích các đảo **khác nhau** ở 4 hàng xóm rồi `+ 1`.

In [34]:
def largest_after_flip(grid):
    pass            # TODO


# Khi làm xong:
# assert largest_after_flip([[1, 0], [0, 1]]) == 3
# assert largest_after_flip([[1, 1], [1, 1]]) == 4
# assert largest_after_flip([[0, 0], [0, 0]]) == 1

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: đếm kích thước và tổng ô đất.
- Mức 2: ô `(2,2)` có kề đất nào theo 4 hướng không?
- Mức 3: đảo `{(0,0),(0,1),(1,1)}` diện tích 3; `(2,0)` và `(2,2)` mỗi ô 1. Đáp án `3`, danh sách `[3, 1, 1]`.

**Bài 2**

- Mức 1: đi từ từng ô đất chưa tính, đánh dấu những ô nối được.
- Mức 2: cột `3` có ba ô `(0,3),(1,3),(2,3)` nối nhau.
- Mức 3: với 8 hướng, đảo nào chạm chéo nhau thì gộp lại (ví dụ `(2,3)` và `(3,4)`).

**Bài 3**

- Mức 1: điều kiện dừng giống mục 7, Cách 2.
- Mức 2: `grid[i][j] = 0` trước khi gọi 4 hướng.
- Mức 3: `return 1 + dfs(i-1,j) + dfs(i+1,j) + dfs(i,j-1) + dfs(i,j+1)`.

**Bài 4**

- Mức 1: vòng ngoài duyệt mọi ô; gặp đất thì loang.
- Mức 2: đánh dấu khi đưa vào hàng đợi, đếm `area`.
- Mức 3: xem mục 9.

**Bài 5**

- Mức 1: gán mỗi đảo một nhãn, lưu diện tích theo nhãn.
- Mức 2: với ô nước, lấy **tập** nhãn của 4 hàng xóm (dùng `set` để không cộng trùng).
- Mức 3: đáp án = max của `1 + tổng area[nhãn]` trên mọi ô nước; nếu không có ô nước thì `m·n`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert` để chạy kiểm tra.

### Bài 1

In [35]:
g = [[1, 1, 0], [0, 1, 0], [1, 0, 1]]
assert (len(g), len(g[0]), sum(map(sum, g))) == (3, 3, 5)
assert max_area_bfs(deepcopy(g)) == 3 and island_areas(g) == [3, 1, 1]
print("Bài 1 OK")

Bài 1 OK


### Bài 2

(a) Đảo A: `(0,0),(0,1),(1,0)` — 3 ô. Đảo B: `(0,3),(0,4),(1,3),(2,3),(2,2)` — 5 ô. Đảo C: `(3,0)` — 1 ô. Đảo D: `(3,4)` — 1 ô. (b) `5`. (c) Với 8 hướng, `(2,3)` chạm chéo `(3,4)` nên D gộp vào B: diện tích `6`; khác vì đường chéo nối thêm đảo.

In [36]:
g = [[1, 1, 0, 1, 1], [1, 0, 0, 1, 0], [0, 0, 1, 1, 0], [1, 0, 0, 0, 1]]
assert island_areas(g) == [5, 3, 1, 1]
assert Solution().maxAreaOfIsland(deepcopy(g)) == 5
assert bug_eight_dirs(deepcopy(g)) == 6
print("Bài 2 OK:", island_areas(g), "| 8 hướng:", bug_eight_dirs(deepcopy(g)))

Bài 2 OK: [5, 3, 1, 1] | 8 hướng: 6


### Bài 3

In [37]:
def max_area_answer(grid):
    m, n = len(grid), len(grid[0])

    def dfs(i, j):
        if not (0 <= i < m and 0 <= j < n) or grid[i][j] == 0:
            return 0
        grid[i][j] = 0
        return 1 + dfs(i - 1, j) + dfs(i + 1, j) + dfs(i, j - 1) + dfs(i, j + 1)

    return max(dfs(i, j) for i in range(m) for j in range(n))


assert max_area_answer(deepcopy(grid1)) == 6 and max_area_answer([[0, 0]]) == 0
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [38]:
class MySolutionAnswer:
    def maxAreaOfIsland(self, grid):
        m, n = len(grid), len(grid[0])
        best = 0
        for i in range(m):
            for j in range(n):
                if grid[i][j] == 1:
                    grid[i][j] = 0
                    area, q = 1, deque([(i, j)])
                    while q:
                        a, b = q.popleft()
                        for dx, dy in ((-1, 0), (1, 0), (0, -1), (0, 1)):
                            x, y = a + dx, b + dy
                            if 0 <= x < m and 0 <= y < n and grid[x][y] == 1:
                                grid[x][y] = 0
                                area += 1
                                q.append((x, y))
                    best = max(best, area)
        return best


for g, w in [(grid1, 6), (grid2, 0), (small, 4)]:
    assert MySolutionAnswer().maxAreaOfIsland(deepcopy(g)) == w
rng = random.Random(4)
for _ in range(300):
    g = random_grid(rng, rng.randint(1, 8), rng.randint(1, 8), 0.5)
    assert MySolutionAnswer().maxAreaOfIsland(deepcopy(g)) == islands_by_sets(g)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

Gán nhãn mỗi đảo, lưu diện tích; với mỗi ô nước, cộng diện tích các **nhãn khác nhau** của 4 hàng xóm, `+ 1` cho ô vừa lật.

In [39]:
def largest_after_flip_answer(grid):
    m, n = len(grid), len(grid[0])
    label = [[0] * n for _ in range(m)]           # 0 = chưa gán / nước
    area = {}
    nid = 0
    for i in range(m):
        for j in range(n):
            if grid[i][j] == 1 and label[i][j] == 0:
                nid += 1
                label[i][j] = nid
                q, cnt = deque([(i, j)]), 1
                while q:
                    a, b = q.popleft()
                    for x, y in neighbors4(a, b, m, n):
                        if grid[x][y] == 1 and label[x][y] == 0:
                            label[x][y] = nid
                            cnt += 1
                            q.append((x, y))
                area[nid] = cnt
    best = max(area.values(), default=0)
    for i in range(m):
        for j in range(n):
            if grid[i][j] == 0:
                near = {label[x][y] for x, y in neighbors4(i, j, m, n) if grid[x][y] == 1}
                best = max(best, 1 + sum(area[k] for k in near))
    return best


def largest_after_flip_oracle(grid):
    """Thử lật từng ô nước rồi chạy lời giải gốc."""
    best = Solution().maxAreaOfIsland(deepcopy(grid))
    for i in range(len(grid)):
        for j in range(len(grid[0])):
            if grid[i][j] == 0:
                g = deepcopy(grid)
                g[i][j] = 1
                best = max(best, Solution().maxAreaOfIsland(g))
    return best


assert largest_after_flip_answer([[1, 0], [0, 1]]) == 3
assert largest_after_flip_answer([[1, 1], [1, 1]]) == 4
assert largest_after_flip_answer([[0, 0], [0, 0]]) == 1
rng = random.Random(55)
for _ in range(600):
    g = random_grid(rng, rng.randint(1, 7), rng.randint(1, 7), rng.choice([0.3, 0.5, 0.7]))
    assert largest_after_flip_answer(g) == largest_after_flip_oracle(g), g
print("Bài 5 OK (đối chiếu oracle 'thử lật từng ô' trên 600 lưới)")

Bài 5 OK (đối chiếu oracle 'thử lật từng ô' trên 600 lưới)

## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (diện tích thành phần liên thông lớn nhất, BFS):**

```python
class Solution:
    def maxAreaOfIsland(self, grid):
        m, n = len(grid), len(grid[0])
        ans = 0
        for i in range(m):
            for j in range(n):
                if grid[i][j] == 1:
                    grid[i][j] = 0                 # đánh dấu ngay
                    area, queue = 1, deque([(i, j)])
                    while queue:
                        x, y = queue.popleft()
                        for dx, dy in ((-1, 0), (1, 0), (0, -1), (0, 1)):
                            a, b = x + dx, y + dy
                            if 0 <= a < m and 0 <= b < n and grid[a][b] == 1:
                                grid[a][b] = 0
                                area += 1
                                queue.append((a, b))
                    ans = max(ans, area)
        return ans
```

**Checklist trước khi submit:**

- [ ] Chỉ **4 hướng** (không đường chéo)?
- [ ] Kiểm tra biên **cả hai phía** `0 <= x < m`, `0 <= y < n`?
- [ ] So sánh với **số nguyên** `1` (không phải `"1"`)?
- [ ] Đánh dấu đã thăm **ngay khi** đưa vào hàng đợi (không có ô nào vào hai lần)?
- [ ] `area` được đặt lại cho **mỗi đảo mới** và `ans = max(...)` sau mỗi đảo?
- [ ] Không có đảo → trả `0` (khởi tạo `ans = 0`)?
- [ ] Nếu dùng đệ quy: nâng `sys.setrecursionlimit` hoặc chuyển sang BFS?
- [ ] Khi tự test: `deepcopy` lưới vì hàm sửa đầu vào; đúng tên `class Solution`, `maxAreaOfIsland`.